# Module 5: DataFrames & Series

## Working with Structured Data in AI

Welcome to Module 5! In this tutorial, you'll learn how to work with structured data using pandas, Python's most powerful data manipulation library.

### Why This Matters for AI Research

Real-world AI data lives in tables:
- **CSV files** from experiments and datasets
- **Database exports** with user interactions
- **API responses** structured as JSON
- **Training datasets** for machine learning models

Pandas lets you manipulate structured data like a programmable spreadsheet. Instead of clicking through Excel, you write code that's:
- **Reproducible** — run the same analysis again and get the same results
- **Scalable** — work with millions of rows
- **Automatable** — turn your analysis into a pipeline

### What You'll Learn

1. Loading data from CSV and JSON files
2. Understanding DataFrame and Series structures
3. Inspecting and exploring your data
4. Selecting specific rows and columns
5. Finding and handling missing data
6. Analyzing distributions and patterns
7. Grouping and aggregating data

### Dataset: LLM Prompt-Response Analysis

We'll work with a real dataset of prompts sent to different AI models (GPT-4, Claude-3, Gemini-Pro), tracking:
- What prompts were asked
- Which model responded
- How many tokens were used
- User ratings of responses
- Categories (coding, analysis, creative, math)

This mirrors real AI research workflows where you analyze model performance across different tasks.

## Setup

First, let's import the libraries we'll need and configure our environment.

In [ ]:
# Environment setup
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

# Core libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Configure matplotlib for inline display
%matplotlib inline

# Display settings for pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("✓ Libraries imported successfully")
print(f"✓ Pandas version: {pd.__version__}")

## 1. Introduction: From Spreadsheets to DataFrames

If you've used Excel or Google Sheets, you already understand tables:
- **Rows** represent individual records (one prompt-response pair)
- **Columns** represent features (model, rating, tokens)
- **Cells** contain values

A **DataFrame** is pandas' version of a table, but with superpowers:
- Operations are **vectorized** (fast on large data)
- You can **filter, transform, and aggregate** with code
- It integrates seamlessly with machine learning libraries

```mermaid
graph LR
    A[Raw Data Files] --> B[pandas DataFrame]
    B --> C[Clean & Transform]
    C --> D[Analyze & Visualize]
    C --> E[Train ML Models]
    
    style B fill:#4CAF50,stroke:#333,stroke-width:2px
```

## 2. Loading Data

The first step in any data analysis is loading your data. Pandas supports many formats:
- **CSV** (comma-separated values) — most common
- **JSON** (JavaScript Object Notation) — common for APIs
- **Excel** — `.xlsx` files
- **SQL** — direct database connections
- **Parquet** — efficient binary format

We'll focus on CSV and JSON since they're most common in AI workflows.

### Loading CSV Files

The `pd.read_csv()` function is your workhorse for loading tabular data.

In [ ]:
# Define path to our dataset
data_path = Path('data/llm_prompts.csv')

# Load the CSV file
df = pd.read_csv(data_path)

print(f"✓ Loaded {len(df)} records from {data_path}")
print(f"✓ Data type: {type(df)}")

> **Key Insight**: `pd.read_csv()` returns a **DataFrame object**. This is pandas' fundamental data structure — a 2D labeled table with rows and columns.

### Advanced Loading Options

You can control exactly how pandas interprets your data:

In [ ]:
# Load with explicit data types and date parsing
df = pd.read_csv(
    data_path,
    dtype={
        'prompt_id': 'int64',
        'model': 'category',  # Categorical saves memory for repeated values
        'category': 'category',
        'temperature': 'float64',
        'tokens_used': 'int64',
        'rating': 'float64'
    },
    parse_dates=['timestamp']  # Automatically convert to datetime
)

print("✓ Data loaded with optimized types")

### Loading JSON Files

API responses often come as JSON. Pandas can handle this too:

In [ ]:
# Example: loading JSON (we'll create a sample)
import json

# Sample JSON data structure
sample_json = [
    {"prompt_id": 1, "model": "gpt-4", "rating": 4.8},
    {"prompt_id": 2, "model": "claude-3", "rating": 4.5},
    {"prompt_id": 3, "model": "gemini-pro", "rating": 4.2}
]

# Load JSON into DataFrame
df_json = pd.read_json(json.dumps(sample_json))

print("JSON loaded as DataFrame:")
print(df_json)

> **Key Insight**: Pandas automatically infers the structure of JSON and creates appropriate columns. For nested JSON, you might need `pd.json_normalize()`.

## 3. DataFrame Anatomy

Let's dissect what a DataFrame actually is.

```mermaid
graph TB
    subgraph DataFrame Structure
        A[DataFrame] --> B[Index<br/>Row labels]
        A --> C[Columns<br/>Column labels]
        A --> D[Values<br/>2D array]
        
        C --> E[Series 1<br/>Column data]
        C --> F[Series 2<br/>Column data]
        C --> G[Series N<br/>Column data]
    end
    
    style A fill:#2196F3,stroke:#333,stroke-width:2px,color:#fff
    style B fill:#FF9800,stroke:#333,stroke-width:1px
    style C fill:#FF9800,stroke:#333,stroke-width:1px
    style D fill:#4CAF50,stroke:#333,stroke-width:1px
```

### Components of a DataFrame

1. **Index** — Row labels (default is 0, 1, 2, ...)
2. **Columns** — Column labels (names from your data)
3. **Values** — The actual data in a 2D array
4. **Series** — Each column is a Series (1D labeled array)

In [ ]:
# Display the index
print("Index (row labels):")
print(df.index)
print()

# Display column names
print("Columns:")
print(df.columns.tolist())
print()

# Get the shape (rows, columns)
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")

### What is a Series?

A **Series** is a single column from a DataFrame. It's a 1D labeled array.

In [ ]:
# Extract a single column as a Series
model_series = df['model']

print(f"Type: {type(model_series)}")
print(f"Length: {len(model_series)}")
print(f"Data type: {model_series.dtype}")
print()
print("First 5 values:")
print(model_series.head())

> **Key Insight**: Every column in a DataFrame is a Series. When you select a single column, you get a Series object. When you select multiple columns, you get a DataFrame.

### Data Types (dtypes)

Each column has a data type that determines what operations you can perform:

| dtype | Meaning | Example |
|-------|---------|--------|
| `int64` | Integer | `42`, `1000` |
| `float64` | Floating point | `3.14`, `4.5` |
| `object` | String/mixed | `"gpt-4"`, `"Hello"` |
| `category` | Categorical | `"coding"`, `"math"` (limited options) |
| `datetime64` | Date/time | `2026-06-01` |
| `bool` | Boolean | `True`, `False` |

In [ ]:
# View data types for all columns
print("Data types per column:")
print(df.dtypes)

## 4. Inspection Methods

Before analyzing data, you need to understand what you're working with. Pandas provides several inspection methods.

```mermaid
graph LR
    A[Load Data] --> B[.head/<br/>.tail]
    B --> C[.info]
    C --> D[.describe]
    D --> E[.value_counts]
    E --> F[.groupby]
    
    style A fill:#9E9E9E,stroke:#333,stroke-width:2px
    style B fill:#2196F3,stroke:#333,stroke-width:2px
    style C fill:#2196F3,stroke:#333,stroke-width:2px
    style D fill:#2196F3,stroke:#333,stroke-width:2px
    style E fill:#FF9800,stroke:#333,stroke-width:2px
    style F fill:#4CAF50,stroke:#333,stroke-width:2px
```

### Peek at the Data: `.head()` and `.tail()`

These methods show you the first or last N rows.

In [ ]:
# View first 5 rows (default)
print("First 5 rows:")
df.head()

In [ ]:
# View last 3 rows
print("Last 3 rows:")
df.tail(3)

> **Key Insight**: Always start with `.head()` to sanity-check your data loaded correctly. Look for unexpected values, wrong types, or formatting issues.

### Structure Overview: `.info()`

This method gives you a compact summary of your DataFrame structure.

In [ ]:
# Get comprehensive info about the DataFrame
df.info()

**What `.info()` tells you:**
- Total rows and columns
- Column names
- Non-null counts (helps spot missing data)
- Data types
- Memory usage

### Statistical Summary: `.describe()`

This generates statistics for numeric columns.

In [ ]:
# Get statistical summary
df.describe()

**What `.describe()` shows:**
- **count** — Number of non-null values
- **mean** — Average value
- **std** — Standard deviation (spread)
- **min** — Minimum value
- **25%, 50%, 75%** — Quartiles (percentiles)
- **max** — Maximum value

In [ ]:
# Include non-numeric columns
df.describe(include='all')

> **Key Insight**: Use `.describe()` to quickly spot outliers, data ranges, and potential issues. For example, if `tokens_used` has a max of 1,000,000 but a mean of 150, you've got an outlier to investigate.

### Quick Dimension Check: `.shape`

The `.shape` attribute returns a tuple: `(rows, columns)`

In [ ]:
rows, cols = df.shape
print(f"Dataset has {rows:,} rows and {cols} columns")
print(f"Total cells: {rows * cols:,}")

## 5. Selecting Data

One of pandas' most powerful features is flexible data selection. You can slice and dice your data in many ways.

### Selecting Single Columns

Two ways to select a column:

In [ ]:
# Method 1: Bracket notation (always works)
models_1 = df['model']

# Method 2: Dot notation (works if column name is valid Python identifier)
models_2 = df.model

print("First 5 models:")
print(models_1.head())

> **Best Practice**: Use bracket notation `df['column']` for consistency, especially when column names have spaces or special characters.

### Selecting Multiple Columns

Pass a list of column names to get a subset DataFrame.

In [ ]:
# Select multiple columns
subset = df[['model', 'rating', 'tokens_used']]

print(f"Subset shape: {subset.shape}")
subset.head()

### Selecting Rows by Position: `.iloc[]`

Use `.iloc[]` for integer-based indexing (like list slicing).

In [ ]:
# First row
print("First row:")
print(df.iloc[0])
print()

# First 5 rows
print("First 5 rows:")
df.iloc[0:5]

In [ ]:
# Select specific rows and columns by position
# Rows 0-4, Columns 1, 3, 5 (model, temperature, rating)
df.iloc[0:5, [1, 3, 5]]

### Selecting Rows by Condition: Boolean Indexing

This is where pandas really shines. You can filter rows based on conditions.

In [ ]:
# Find all rows where rating > 4.5
high_rated = df[df['rating'] > 4.5]

print(f"Found {len(high_rated)} high-rated responses")
high_rated.head()

In [ ]:
# Multiple conditions: high rating AND coding category
high_rated_coding = df[(df['rating'] > 4.5) & (df['category'] == 'coding')]

print(f"Found {len(high_rated_coding)} high-rated coding responses")
high_rated_coding.head()

> **Key Insight**: Use `&` for AND, `|` for OR, and `~` for NOT when combining conditions. Always wrap each condition in parentheses.

In [ ]:
# Filter for GPT-4 OR Claude-3
major_models = df[df['model'].isin(['gpt-4', 'claude-3'])]

print(f"GPT-4 and Claude-3 prompts: {len(major_models)}")
print(major_models['model'].value_counts())

## 6. Finding Missing Data

Missing data is common in real-world datasets. It can cause errors in machine learning if not handled properly.

### Detecting Missing Values

Pandas represents missing data as `NaN` (Not a Number) or `None`.

In [ ]:
# Check for missing values (returns True/False for each cell)
print("Missing values map (first 5 rows):")
df.isna().head()

In [ ]:
# Count missing values per column
missing_counts = df.isna().sum()

print("Missing values per column:")
print(missing_counts)
print()

# Only show columns with missing data
has_missing = missing_counts[missing_counts > 0]
print("Columns with missing data:")
print(has_missing)

### Visualizing Missing Data

Let's create a bar chart to see where data is missing.

In [ ]:
# Visualize missing data
missing_counts = df.isna().sum()
columns_with_missing = missing_counts[missing_counts > 0]

if len(columns_with_missing) > 0:
    plt.figure(figsize=(10, 5))
    columns_with_missing.plot(kind='bar', color='#FF5252')
    plt.title('Missing Values by Column', fontsize=14, fontweight='bold')
    plt.xlabel('Column', fontsize=12)
    plt.ylabel('Count of Missing Values', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("✓ No missing values found in dataset")

> **Key Insight**: The `rating` column has 3 missing values. This is important for analysis — we'll need to decide whether to drop these rows, fill them, or exclude them from rating calculations.

### Why Missing Data Matters for ML

Most machine learning algorithms cannot handle missing values. You have three options:

1. **Drop rows with missing data** — simple but loses information
2. **Fill with a value** — mean, median, mode, or domain-specific default
3. **Imputation** — predict missing values using other features

The choice depends on:
- How much data is missing
- Whether the missingness is random
- What the missing value means in context

In [ ]:
# Example: View rows with missing ratings
missing_rating = df[df['rating'].isna()]

print(f"Rows with missing rating: {len(missing_rating)}")
missing_rating[['prompt_id', 'model', 'category', 'rating']]

## 7. Value Counts & Distributions

Before building models, you need to understand the distribution of your data.

### Counting Categorical Values

Use `.value_counts()` to see the frequency of each unique value.

In [ ]:
# Count how many prompts each model handled
model_counts = df['model'].value_counts()

print("Prompts per model:")
print(model_counts)
print()

# As percentages
print("As percentages:")
print(df['model'].value_counts(normalize=True) * 100)

In [ ]:
# Visualize model distribution
plt.figure(figsize=(10, 6))
model_counts = df['model'].value_counts()
model_counts.plot(kind='bar', color=['#4CAF50', '#2196F3', '#FF9800'])
plt.title('Distribution of Prompts by Model', fontsize=14, fontweight='bold')
plt.xlabel('Model', fontsize=12)
plt.ylabel('Number of Prompts', fontsize=12)
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nTotal prompts: {model_counts.sum()}")

> **Key Insight**: The dataset is roughly balanced across models. If one model had 80% of prompts, we'd need to be careful about drawing conclusions — results might just reflect that model's dominance.

In [ ]:
# Check category distribution
category_counts = df['category'].value_counts()

print("Prompts per category:")
print(category_counts)

# Visualize
plt.figure(figsize=(10, 6))
category_counts.plot(kind='bar', color='#9C27B0')
plt.title('Distribution of Prompts by Category', fontsize=14, fontweight='bold')
plt.xlabel('Category', fontsize=12)
plt.ylabel('Number of Prompts', fontsize=12)
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Numeric Distributions: Histograms

Histograms show you the distribution shape for numeric data.

In [ ]:
# Histogram of tokens used
plt.figure(figsize=(10, 6))
df['tokens_used'].hist(bins=20, color='#00BCD4', edgecolor='black')
plt.title('Distribution of Tokens Used', fontsize=14, fontweight='bold')
plt.xlabel('Tokens Used', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.axvline(df['tokens_used'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["tokens_used"].mean():.1f}')
plt.axvline(df['tokens_used'].median(), color='orange', linestyle='--', linewidth=2, label=f'Median: {df["tokens_used"].median():.1f}')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean tokens: {df['tokens_used'].mean():.1f}")
print(f"Median tokens: {df['tokens_used'].median():.1f}")
print(f"Std deviation: {df['tokens_used'].std():.1f}")

> **Key Insight**: This distribution shows most prompts use 100-200 tokens, with a long tail of higher-token responses. This is typical for LLM data — short prompts are common, but some generate lengthy responses.

In [ ]:
# Rating distribution (excluding missing values)
plt.figure(figsize=(10, 6))
df['rating'].dropna().hist(bins=15, color='#FFC107', edgecolor='black')
plt.title('Distribution of User Ratings', fontsize=14, fontweight='bold')
plt.xlabel('Rating', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.axvline(df['rating'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["rating"].mean():.2f}')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean rating: {df['rating'].mean():.2f}")
print(f"Median rating: {df['rating'].median():.2f}")
print(f"Rating range: {df['rating'].min():.1f} - {df['rating'].max():.1f}")

## 8. Grouping & Aggregation

The real power of pandas comes from grouping and aggregating data. This is the **split-apply-combine** pattern:

```mermaid
graph LR
    A[Full Dataset] --> B[Split by Category]
    B --> C1[Group: Coding]
    B --> C2[Group: Analysis]
    B --> C3[Group: Creative]
    B --> C4[Group: Math]
    
    C1 --> D[Apply Function<br/>mean, sum, count]
    C2 --> D
    C3 --> D
    C4 --> D
    
    D --> E[Combine Results]
    
    style A fill:#9E9E9E,stroke:#333,stroke-width:2px
    style B fill:#FF9800,stroke:#333,stroke-width:2px
    style D fill:#2196F3,stroke:#333,stroke-width:2px
    style E fill:#4CAF50,stroke:#333,stroke-width:2px
```

### Basic GroupBy

Group data by a categorical column, then compute statistics.

In [ ]:
# Average tokens used by model
tokens_by_model = df.groupby('model')['tokens_used'].mean()

print("Average tokens by model:")
print(tokens_by_model.sort_values(ascending=False))

In [ ]:
# Multiple aggregations at once
model_stats = df.groupby('model')['tokens_used'].agg([
    ('count', 'count'),
    ('mean', 'mean'),
    ('median', 'median'),
    ('std', 'std'),
    ('min', 'min'),
    ('max', 'max')
]).round(2)

print("Token statistics by model:")
model_stats

> **Key Insight**: Gemini-Pro tends to generate longer responses (higher mean/median tokens) compared to GPT-4 and Claude-3. This could reflect the model's verbosity or the types of prompts it received.

### Multi-Column Aggregation

In [ ]:
# Group by category and calculate multiple metrics
category_summary = df.groupby('category').agg({
    'prompt_id': 'count',        # Count of prompts
    'tokens_used': 'mean',       # Average tokens
    'rating': 'mean',            # Average rating (ignoring NaN)
    'temperature': 'mean'        # Average temperature
}).round(2)

# Rename columns for clarity
category_summary.columns = ['num_prompts', 'avg_tokens', 'avg_rating', 'avg_temperature']

print("Summary by category:")
category_summary.sort_values('avg_rating', ascending=False)

In [ ]:
# Visualize average rating by category
plt.figure(figsize=(10, 6))
category_avg_rating = df.groupby('category')['rating'].mean().sort_values(ascending=False)
category_avg_rating.plot(kind='bar', color='#E91E63')
plt.title('Average Rating by Category', fontsize=14, fontweight='bold')
plt.xlabel('Category', fontsize=12)
plt.ylabel('Average Rating', fontsize=12)
plt.xticks(rotation=0)
plt.axhline(df['rating'].mean(), color='black', linestyle='--', linewidth=2, label=f'Overall Mean: {df["rating"].mean():.2f}')
plt.ylim(3.5, 5.0)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Multi-Level GroupBy

You can group by multiple columns simultaneously.

In [ ]:
# Group by model AND category
model_category_rating = df.groupby(['model', 'category'])['rating'].mean().round(2)

print("Average rating by model and category:")
print(model_category_rating)

In [ ]:
# Unstack for better readability (pivot table style)
model_category_pivot = df.groupby(['model', 'category'])['rating'].mean().unstack(fill_value=0).round(2)

print("Rating matrix (model × category):")
model_category_pivot

> **Key Insight**: You can now compare model performance across different task types. For example, GPT-4 might excel at coding tasks while Claude-3 performs better on analysis.

## 9. Lab: Dataset Detective

Now it's your turn! Apply everything you've learned to explore the LLM prompts dataset.

### Your Mission

Complete these analysis tasks:

1. **Load & Inspect**
   - Load the data (already done above)
   - Display shape, info, and statistical summary
   - Identify column types

2. **Missing Data Analysis**
   - Find all columns with missing values
   - Calculate percentage of missing data
   - Visualize missing data distribution

3. **Distribution Analysis**
   - Create value_counts for 'model' and 'category'
   - Plot histogram of 'tokens_used'
   - Plot histogram of 'rating'

4. **Comparative Analysis**
   - Calculate average tokens by model
   - Calculate average rating by category
   - Create a pivot table of model × category with average ratings

5. **Write Your Findings**
   - Summarize key insights in markdown cells
   - What patterns did you discover?
   - What questions would you investigate next?

### Task 1: Load & Inspect

In [ ]:
# Your code here
# Display df.shape, df.info(), and df.describe()


**Your observations:**
- Number of rows: ___
- Number of columns: ___
- Column types: ___
- Any surprises? ___

### Task 2: Missing Data Analysis

In [ ]:
# Your code here
# Find and visualize missing values


**Your observations:**
- Which columns have missing data? ___
- What percentage is missing? ___
- Should we drop these rows, fill values, or keep them? ___

### Task 3: Distribution Analysis

In [ ]:
# Your code here
# Create value_counts and histograms


**Your observations:**
- Is the dataset balanced across models? ___
- Is the dataset balanced across categories? ___
- What's the typical token usage range? ___
- What's the typical rating range? ___

### Task 4: Comparative Analysis

In [ ]:
# Your code here
# GroupBy analysis and pivot tables


**Your observations:**
- Which model uses the most tokens on average? ___
- Which category has the highest ratings? ___
- Which model performs best on coding tasks? ___
- Any surprising patterns? ___

### Task 5: Your Key Findings

Write a summary of your analysis here. Include:
- 3-5 key insights
- Patterns you discovered
- Questions for further investigation

**Example:**
> I found that creative prompts tend to use more tokens than math prompts, suggesting that generative tasks produce longer responses. GPT-4 has the highest ratings for coding tasks (4.85 average) but Claude-3 performs better on analysis tasks. I'd like to investigate whether temperature settings correlate with rating quality.

---

**Your findings:**

_(Write your summary here)_

## Summary & Key Takeaways

Congratulations! You've learned the fundamentals of working with structured data in pandas.

### What You Learned

1. **Loading Data**
   - `pd.read_csv()` and `pd.read_json()` for importing data
   - Specifying dtypes and parsing dates

2. **DataFrame Anatomy**
   - Structure: index (rows), columns, values
   - Series: a single column
   - Data types: int64, float64, object, category, datetime64

3. **Inspection Methods**
   - `.head()` and `.tail()` — peek at data
   - `.info()` — structure and types
   - `.describe()` — statistical summary
   - `.shape` — dimensions

4. **Selection**
   - Single column: `df['column']`
   - Multiple columns: `df[['col1', 'col2']]`
   - Rows by position: `.iloc[]`
   - Rows by condition: Boolean indexing

5. **Missing Data**
   - `.isna()` to detect missing values
   - `.isna().sum()` to count missing values
   - Visualization and handling strategies

6. **Distributions**
   - `.value_counts()` for categorical data
   - `.hist()` for numeric data
   - Understanding data before modeling

7. **Grouping & Aggregation**
   - `.groupby()` for split-apply-combine
   - Aggregation functions: mean, sum, count, etc.
   - Multi-level grouping and pivot tables

### Why This Matters for AI

Every AI project starts with data exploration:
- **Before training** — understand your data distribution, find class imbalances, spot outliers
- **During experiments** — track metrics across model variants, analyze failure cases
- **After deployment** — monitor performance metrics, detect data drift

Pandas is your primary tool for all of these tasks.

## Try It Yourself: Extension Exercises

Ready for more practice? Try these challenges:

### Exercise 1: Temperature Analysis
Analyze how the `temperature` parameter affects response characteristics:
- Plot temperature distribution
- Calculate average tokens_used for different temperature ranges (0-0.3, 0.4-0.6, 0.7-1.0)
- Does higher temperature correlate with longer responses?

### Exercise 2: Timestamp Analysis
The `timestamp` column contains date/time information:
- Extract the hour from timestamps
- Plot distribution of prompts by hour of day
- Are certain types of prompts more common at certain times?

### Exercise 3: Model Comparison Report
Create a comprehensive comparison of the three models:
- Average rating per model
- Token efficiency (rating per token)
- Performance by category
- Write a recommendation: which model would you use for which tasks?

### Exercise 4: Custom Analysis
Come up with your own research question and answer it using the dataset. Examples:
- Do creative prompts get lower ratings than technical ones?
- Is there a relationship between tokens_used and rating?
- Which model has the most consistent ratings (lowest std deviation)?

Document your question, analysis code, and findings.

## Next Steps

In the next module, you'll learn:
- **Data Cleaning** — handling missing values, removing duplicates, fixing inconsistencies
- **Feature Engineering** — creating new columns from existing data
- **Advanced Filtering** — complex queries and transformations
- **Merging DataFrames** — combining data from multiple sources

These skills build directly on what you learned here. Mastering DataFrames is essential for every AI researcher and data scientist.

### Resources

- [Pandas Official Documentation](https://pandas.pydata.org/docs/)
- [Pandas Cheat Sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf)
- [Real Python Pandas Tutorials](https://realpython.com/learning-paths/pandas-data-science/)

Keep exploring, and remember: **every expert was once a beginner who didn't give up**. 🚀